# Load YOLO segmentation datasets with compact masks

[Open in Colab](https://colab.research.google.com/github/roboflow/supervision/blob/refs/pull/2834/head/examples/compact_mask/yolo_loading.ipynb)

Self-contained CPU demonstration for [Supervision #2834](https://github.com/roboflow/supervision/pull/2834). No model, Roboflow account, or dataset download is required. The setup installs the PR revision; restart the runtime if Supervision was already imported.

The Linux benchmark creates 16 images with 8 small polygons each, compares default dense loading with `compact_masks=True`, and checks exact pixel, box and class-ID equality. Each mode is measured in three fresh processes.

In [ ]:
%pip install -q "git+https://github.com/roboflow/supervision.git@refs/pull/2834/head"

In [ ]:
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path

url = (
    "https://raw.githubusercontent.com/roboflow/supervision/refs/pull/2834/head/"
    "examples/compact_mask/benchmark_yolo.py"
)
with tempfile.TemporaryDirectory() as directory:
    script = Path(directory) / "benchmark_yolo.py"
    urllib.request.urlretrieve(url, script)
    subprocess.run([sys.executable, str(script)], check=True)

## Interpret the output

- `mask_buffer_bytes` counts dense array bytes or compact RLE/crop-shape/offset array bytes, excluding Python-object and allocator overhead.
- `peak_rss_mib` is Linux process peak RSS, including imports and input generation; it is a different measurement from retained mask storage.
- `load_seconds` excludes imports and input generation. Pixel equality runs separately so decoding does not inflate measured peak RSS.
- These sparse synthetic polygons compress well; fragmented masks can compress poorly. This is compact eager loading: each polygon still temporarily uses a full-frame mask. Calling `to_dense()` allocates dense masks again.

Related work: [lazy annotations #1484](https://github.com/roboflow/supervision/pull/1484), [compact ingestion #2367](https://github.com/roboflow/supervision/pull/2367), [polygon crop bounds #2678](https://github.com/roboflow/supervision/pull/2678), and [RF-DETR lazy loading #851](https://github.com/roboflow/rf-detr/pull/851). This demo uses the existing CompactMask implementation and does not replace RF-DETR's fully lazy loader.